In [37]:
import torch

In [38]:
import numpy as np
import pandas as pd

In [39]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv("dataset/final_processed_data.csv") #load dataset

embedding_cache = torch.load("chemberta_smiles_embeddings.pt") # load embeddings

emb_s1 = torch.stack([embedding_cache[s] for s in df["SMILES1"]])
emb_s2 = torch.stack([embedding_cache[s] for s in df["SMILES2"]])

In [41]:
combined_embeddings = []

for e1, e2 in zip(emb_s1, emb_s2):
    combined_embeddings.append(np.concatenate((e1, e2)))

combined_embeddings = np.array(combined_embeddings)

In [42]:
sample_size = 5000
np.random.seed(42)
indices = np.random.choice(len(combined_embeddings), sample_size, replace=False)

emb_sample = combined_embeddings[indices]

In [43]:
pca = PCA(n_components=50) # reduce to 50 dimensions for clustering
emb_pca = pca.fit_transform(emb_sample)

In [44]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k = 10  # try multiple values

kmeans = KMeans(n_clusters=k, random_state=42)
labels = kmeans.fit_predict(emb_pca)

score = silhouette_score(emb_pca, labels)

print(f"Silhouette Score (k={k}): {score:.4f}")

Silhouette Score (k=10): 0.0813


In [45]:
for k in [5, 10, 15, 20]:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(emb_pca)
    score = silhouette_score(emb_pca, labels)
    print(f"k={k}, Silhouette Score={score:.4f}")

k=5, Silhouette Score=0.0821
k=10, Silhouette Score=0.0813
k=15, Silhouette Score=0.0744
k=20, Silhouette Score=0.0722
